# 04 — Training Walk-Forward CV

Train TCN+GRU folds with train-only scaling, HMM diagnostics, and artifact export.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR

In [ ]:
import pandas as pd
from google.colab import runtime
from yenibot.experiments import run_experiment_matrix

labeled = pd.read_parquet(f'{DATA_DIR}/processed/labeled_1h.parquet')
print('Training rows:', len(labeled))
print('Experiment mode:', cfg.get('experiments', {}).get('mode', 'staged'))
print('Control profile:', cfg.get('experiments', {}).get('control_profile'))
print('Candidate profiles:', cfg.get('experiments', {}).get('candidate_profiles', []))
print('Full CV mode:', cfg.get('experiments', {}).get('full_cv_profiles', 'auto'))
print('Always-full profiles:', cfg.get('experiments', {}).get('always_full_profiles', []))
print('Max auto full candidates:', cfg.get('experiments', {}).get('max_auto_full_candidates'))
print('Seed:', cfg['project']['random_seed'], 'deterministic:', cfg['project'].get('deterministic', False))

try:
    result = run_experiment_matrix(
        labeled,
        cfg,
        checkpoint_dir=CHECKPT_DIR,
    )
    print('Experiment run:', result['run_id'])
    print('Experiment dir:', result['run_dir'])
    display(result['comparison'])
    print('Decision:', result['decision'])
finally:
    runtime.unassign()


In [ ]:
# Runtime is released once, after the experiment matrix cell finishes.
# Outputs are profile-isolated under: {CHECKPT_DIR}/experiments/{run_id}/{profile}/{fold_scope}/
